# 05 · Fine-tune the front end  (P7)

One overnight run. Release the **top 6** transformer layers of the front end
and train them jointly with the head at a very low learning rate.

Why top-6 rather than a full unfreeze: it captures most of the gain, fits
comfortably in 16 GB with mixed precision and gradient accumulation, and will
not run out of memory at 2am the night before.

**Keep the P2 checkpoints.** If this run disappoints, they are the fallback,
and a disappointing fine-tune is a normal outcome rather than a disaster.

In [ ]:
# --- Colab setup -----------------------------------------------------------
# Run this first in every notebook.  Idempotent.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    # Keep the repo and all caches on Drive so a disconnect does not cost you
    # the feature extraction pass.
    PROJECT = Path("/content/drive/MyDrive/voice-integrity")
    if not PROJECT.exists():
        raise SystemExit(
            f"Upload or clone the repo to {PROJECT} first.\n"
            "  !git clone <your-repo-url> /content/drive/MyDrive/voice-integrity"
        )
else:
    PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))

# Repo-local model cache.  Set BEFORE importing transformers, or it will use
# the default location and the cache will not be portable to the demo machine.
os.environ["HF_HOME"] = str(PROJECT / "cache" / "huggingface")
os.environ["TORCH_HOME"] = str(PROJECT / "cache" / "torch")
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

print("project:", PROJECT)
print("python :", sys.version.split()[0])

In [ ]:
if IN_COLAB:
    !pip install -q transformers speechbrain soundfile librosa pydantic pyyaml cryptography wandb
    !apt-get -qq install -y ffmpeg libopencore-amrnb-dev > /dev/null

# AMR-NB encoding is the one that silently goes missing.  If this prints
# nothing, your mobile-codec augmentation does nothing and the whole
# codec-robustness result quietly evaporates.
!ffmpeg -hide_banner -encoders 2>/dev/null | grep -i amr || echo "AMR-NB ENCODER MISSING"

In [ ]:
import torch
print("cuda available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device         :", torch.cuda.get_device_name(0))
    print("memory         : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

This path reads **audio**, not cached features — the front end is being
trained now, so its outputs change every step. That is the whole reason it
costs a night instead of two minutes.

In [ ]:
from vif.common.config import load_config
from vif.data.manifests import read_manifest
from vif.models.heads import load_checkpoint

config = load_config("configs")
train_items = read_manifest("data/manifests/asvspoof19la_train.jsonl")
dev_items   = read_manifest("data/manifests/asvspoof19la_dev.jsonl")

# Start from the codec-robust head rather than from scratch.
head, meta = load_checkpoint(
    "models/checkpoints/codec_robust.pt", config.model.head,
    feat_dim=config.model.frontend.hidden_dim,
    expect_window=config.model.audio.window_samples,
)
print("starting from:", meta.get("condition"), "epoch", meta.get("epoch"))

In [ ]:
from vif.train.finetune import FinetuneConfig, finetune

ft_config = FinetuneConfig(
    epochs=4,
    batch_size=4,               # small batch ...
    grad_accum_steps=8,         # ... with accumulation to an effective 32
    frontend_lr=1e-6,           # deliberately tiny: this is a fine-tune
    head_lr=5e-5,
    mixed_precision=True,
    gradient_checkpointing=True,  # slower per step, much less memory
)

frontend, head, best_eer = finetune(
    config, train_items, dev_items, head,
    ft_config=ft_config,
    device=DEVICE,
    checkpoint_path="models/checkpoints/finetuned.pt",
    augment=True,
)
print(f"\nbest dev EER {best_eer*100:.2f}%")

### If this OOMs

In order of preference:

1. Lower `unfreeze_top_n` in `configs/model.yaml` from 6 to 4.
2. Keep `gradient_checkpointing=True` (it is already on).
3. Drop `batch_size` to 2 and raise `grad_accum_steps` to 16 — the effective
   batch is unchanged.

Do **not** disable mixed precision to fix memory; it makes things worse.

In [ ]:
# Notebook 04 cannot score this model: its caches hold features from the frozen
# front end, and the fine-tuned head was trained on the released layers.
# To evaluate it, re-extract the eval caches with this front end.  To serve it,
# point head.checkpoint in configs/model.yaml at finetuned.pt - the detector
# loads frontend_top_layers.pt beside it - and refit calibration on dev scores
# from this model first.
import json
print(json.dumps({"best_eer": best_eer, "note": "P2 checkpoints retained as fallback"}, indent=2))